# Day 187 — Process Mining with pm4py (Part 1)
### Month 11: Data Engineering Portfolio | LogiTrack India — Shipment Process Log

**Dataset:** `logitrack_process_log_RAW.csv` — seed=187, 150 shipment cases, 1186 events, 9 activities
**Tool:** `pm4py` (open-source, AGPL v3 license — free for personal/portfolio use; commercial SaaS use requires either open-sourcing your app or a paid license from processintelligence.solutions)
**Scope:** Event log construction → variant analysis → process discovery (DFG + Petri net). Conformance checking and KPI dashboards are Part 2 (Day 188).

---


## Setup — run this first

In [3]:
# Run once per Colab session. --break-system-packages is only needed on some local Linux setups; harmless on Colab.
!pip install pm4py --quiet
!apt-get install -y graphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [4]:
import pandas as pd
import numpy as np
import pm4py
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("pm4py version:", pm4py.__version__)




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




pm4py version: 2.7.23.1


## SECTION 1: Raw Data — DO NOT MODIFY

This cell **generates and locks** the raw event log. Treat it exactly like a CSV a client handed you —
read from it, never edit it in place. Every transformation happens in a separate DataFrame/log object downstream.

**Columns:**
| Column | Meaning |
|---|---|
| `case_id` | The shipment being tracked (this is the *process instance* / "trace") |
| `activity` | The step that occurred (this is the *event* / "task") |
| `timestamp` | When it occurred |
| `resource` | Who/what performed it (agent, warehouse, system) |
| `carrier` | Delivery carrier for this shipment |
| `region` | Origin warehouse region |


In [5]:
# ============================================================
# LOCKED RAW DATA — DO NOT MODIFY THIS CELL
# LogiTrack India Shipment Process Log | seed=187 | 150 cases
# ============================================================
np.random.seed(187)

N_CASES = 150
CARRIERS = ["BlueDart", "Delhivery", "Ecom Express", "XpressBees", "DTDC"]
AGENTS = [f"Agent_{i:02d}" for i in range(1, 13)]
REGIONS = ["North", "South", "East", "West", "Central"]

CARRIER_SUCCESS_RATE = {
    "BlueDart": 0.90, "Delhivery": 0.85, "Ecom Express": 0.80,
    "XpressBees": 0.62, "DTDC": 0.78
}

from datetime import datetime, timedelta
rows = []
start_base = datetime(2026, 4, 1, 8, 0, 0)

def generate_case(case_num):
    case_id = f"SHIP{case_num:04d}"
    carrier = np.random.choice(CARRIERS)
    region = np.random.choice(REGIONS)
    agent = np.random.choice(AGENTS)
    success_rate = CARRIER_SUCCESS_RATE[carrier]
    t = start_base + timedelta(days=int(np.random.uniform(0, 30)), hours=int(np.random.uniform(0, 10)))
    state = {"t": t}

    def add(activity, hours_gap, resource=None):
        state["t"] = state["t"] + timedelta(hours=float(hours_gap))
        rows.append({"case_id": case_id, "activity": activity, "timestamp": state["t"],
                     "resource": resource if resource else agent, "carrier": carrier, "region": region})

    add("Order Placed", np.random.uniform(0, 1), resource="System")
    add("Packed", np.random.uniform(2, 8), resource=f"Warehouse_{region}")
    add("Shipped", np.random.uniform(1, 4))
    add("In Transit", np.random.uniform(6, 30))
    add("Out for Delivery", np.random.uniform(4, 20))

    delivered = np.random.random() < success_rate
    attempts = 1
    while not delivered:
        add("Delivery Attempted", np.random.uniform(1, 3))
        add("Delivery Failed", np.random.uniform(0.5, 2))
        attempts += 1
        if attempts > 3:
            add("Returned to Warehouse", np.random.uniform(2, 6))
            delivered = None
            break
        add("Out for Delivery", np.random.uniform(12, 24))
        delivered = np.random.random() < success_rate

    if delivered:
        add("Delivery Attempted", np.random.uniform(1, 3))
        add("Delivered", np.random.uniform(0.2, 1.5))

for case_num in range(1, N_CASES + 1):
    generate_case(case_num)

raw_df = pd.DataFrame(rows)
raw_df.to_csv("logitrack_process_log_RAW.csv", index=False)
print("Raw log shape:", raw_df.shape)
raw_df.head(10)


Raw log shape: (1186, 6)


,case_id,activity,timestamp,resource,carrier,region
0,SHIP0001,Order Placed,2026-04-02 13:30:54.583190,System,XpressBees,East
1,SHIP0001,Packed,2026-04-02 21:13:43.538842,Warehouse_East,XpressBees,East
2,SHIP0001,Shipped,2026-04-02 22:18:01.134795,Agent_11,XpressBees,East
3,SHIP0001,In Transit,2026-04-03 22:05:03.537379,Agent_11,XpressBees,East
4,SHIP0001,Out for Delivery,2026-04-04 08:04:33.181936,Agent_11,XpressBees,East
5,SHIP0001,Delivery Attempted,2026-04-04 09:06:19.459303,Agent_11,XpressBees,East
6,SHIP0001,Delivery Failed,2026-04-04 10:34:46.557643,Agent_11,XpressBees,East
7,SHIP0001,Out for Delivery,2026-04-05 02:52:56.873376,Agent_11,XpressBees,East
8,SHIP0001,Delivery Attempted,2026-04-05 05:10:55.834183,Agent_11,XpressBees,East
9,SHIP0001,Delivery Failed,2026-04-05 06:41:43.043249,Agent_11,XpressBees,East


## SECTION 2: Concept Notes

### What is process mining?
Process mining sits between data mining and BPM (business process management). Instead of asking
"what does the process diagram say should happen," it asks **"what does the event data say actually
happened."** You reconstruct the real process — with all its shortcuts, rework loops, and delays — directly
from system logs (ERP, CRM, ticketing, logistics tracking).

### The three columns every event log needs
| Concept | Column here | Definition |
|---|---|---|
| **Case ID** | `case_id` | One instance of the process (one shipment, one order, one support ticket) |
| **Activity** | `activity` | One step within that instance |
| **Timestamp** | `timestamp` | When the step happened — this is what lets pm4py *order* events into a sequence |

A **trace** is the full ordered sequence of activities for one case, e.g.
`Order Placed → Packed → Shipped → In Transit → Out for Delivery → Delivered`.

### Variants
Not every case follows the same path. A **variant** is a distinct *sequence* of activities. If 113 of 150
shipments follow the exact 7-step happy path and 37 detour through `Delivery Failed` loops, you have at
least 2 variants — and the split tells you what fraction of your process is "as designed" vs "exception
handling." This is the single fastest health check on any operational process.

### Directly-Follows Graph (DFG)
The DFG is the simplest process discovery output: for every pair of activities (A, B), count how often B
happens *immediately* after A somewhere in the log. It's a weighted graph, not a strict model — it doesn't
tell you about loops or choices the way a formal model does, but it's the fastest way to see where volume
flows and where it bottlenecks.

### Petri nets (formal process models)
A Petri net is a mathematically precise process model made of:
- **Places** (circles) — states/conditions between activities
- **Transitions** (rectangles) — the activities themselves
- **Arcs** — directed connections between places and transitions
- **Tokens** — mark which state a given case is currently in

Unlike a DFG, a Petri net can correctly represent parallelism, choice, and loops, and it becomes the
input for **conformance checking** (Day 188) — automatically flagging cases that deviate from the
discovered model.

### Discovery algorithms (why we use Inductive Miner)
- **Alpha Miner** — the original academic algorithm; fast, but very fragile with noise and rarely used
  in production.
- **Inductive Miner** — the industry-standard default in pm4py; guarantees a *sound* model (no
  deadlocks) even on noisy real-world logs. This is what `pm4py.discover_petri_net_inductive()` uses,
  and what you should default to on client work unless told otherwise.
- **Heuristics Miner** — good for very noisy logs where you want a graph, not a strict formal model.

### Real-world use (why clients pay for this)
- **Logistics (this dataset):** find which carriers cause rework loops (repeated `Delivery Attempted →
  Delivery Failed`), which is a direct cost driver.
- **Finance/procurement:** find purchase orders that skip approval steps (compliance risk).
- **Healthcare:** find patient pathways that deviate from clinical guidelines.
- **Customer support:** find ticket loops that never reach resolution.

### Freelance/interview framing
If a client asks "why not just build a flowchart," the answer is: a flowchart is what *someone believes*
the process is. Process mining shows what the process *actually does*, discovered directly from
timestamped log data — no interviews, no assumptions, no stale documentation.


## SECTION 3: Practice Guide — Tasks

Complete each task in the blank cell below it. Use only `pandas` and `pm4py` functions introduced above —
no forward references to conformance checking or KPI dashboards (that's Day 188).

**Total: 90 points + 10 bonus stars**


### Task 1 — Extract & Profile (15 pts)
Convert `raw_df` into a pm4py-compatible event log and print a profile:
1. Number of raw rows
2. Number of unique cases
3. Number of unique activities (and list them)
4. Date range (min → max timestamp)
5. Average events per case (rows / cases)

**Hint:** `pm4py.format_dataframe(df, case_id=..., activity_key=..., timestamp_key=...)` then
`pm4py.convert_to_event_log(...)`.


In [6]:
# ── Task 1: Extract & Profile ───────────────────────────────────────────────
# Goal: convert raw_df to pm4py event log and print a profile
# Method: pm4py.format_dataframe → pm4py.convert_to_event_log → profile summaries

# Copy raw data to a working DataFrame
df = raw_df.copy()

# 1. Format for pm4py (rename/tag the three required columns)
log_df = pm4py.format_dataframe(
    df,
    case_id='case_id',
    activity_key='activity',
    timestamp_key='timestamp'
)

# 2. Convert to a pm4py EventLog object
event_log = pm4py.convert_to_event_log(log_df)

# 3. Profile prints
print("Raw rows:", len(df))
print("Unique cases:", len(event_log))
unique_activities = set(df['activity'])
print("Unique activities:", len(unique_activities))
print("Activities:", sorted(unique_activities))
print("Date range:", df['timestamp'].min(), "->", df['timestamp'].max())
print("Avg events/case:", round(len(df) / len(event_log), 2))

Raw rows: 1186
Unique cases: 150
Unique activities: 9
Activities: ['Delivered', 'Delivery Attempted', 'Delivery Failed', 'In Transit', 'Order Placed', 'Out for Delivery', 'Packed', 'Returned to Warehouse', 'Shipped']
Date range: 2026-04-01 09:07:14.786369 -> 2026-05-03 11:29:46.114111
Avg events/case: 7.91


### Task 2 — Variant Analysis (20 pts)
1. Get all trace variants and their counts.
2. Identify the single most common variant and its % share of all cases.
3. Count total number of unique variants.
4. Count how many cases contain a `Delivery Failed` event anywhere (these are your "deviant"/rework
   cases) and their % of total cases.

**Hint:** `from pm4py.statistics.variants.log import get as variants_get`, then
`variants_get.get_variants(event_log)` returns `{variant_tuple: [list of traces]}`.


In [7]:
# ── Task 2: Variant Analysis ───────────────────────────────────────────────
# Goal: get variant counts, top variant, deviant cases with Delivery Failed
# Method: pm4py.statistics.variants.log.get → count, filter for 'Delivery Failed'

from pm4py.statistics.variants.log import get as variants_get

variant_dict = variants_get.get_variants(event_log)
variant_counts = {k: len(v) for k, v in variant_dict.items()}
sorted_variants = sorted(variant_counts.items(), key=lambda x: x[1], reverse=True)

top_variant, top_count = sorted_variants[0]
print("Total unique variants:", len(sorted_variants))
print("Top variant:", top_variant)
print(f"Top variant share: {top_count} cases ({round(100*top_count/len(event_log),1)}%)")

# Count cases that have at least one 'Delivery Failed' event
deviant_cases = df[df['activity'] == 'Delivery Failed']['case_id'].nunique()
print(f"Deviant (rework) cases: {deviant_cases} ({round(100*deviant_cases/len(event_log),1)}%)")

Total unique variants: 4
Top variant: ('Order Placed', 'Packed', 'Shipped', 'In Transit', 'Out for Delivery', 'Delivery Attempted', 'Delivered')
Top variant share: 113 cases (75.3%)
Deviant (rework) cases: 37 (24.7%)


### Task 3 — Process Discovery (20 pts)
1. Discover a Petri net using the **Inductive Miner** (`pm4py.discover_petri_net_inductive`). Print the
   number of places, transitions, and arcs.
2. Discover the **DFG** (`pm4py.discover_dfg`). Print the number of edges and the top 5 directly-follows
   relations by frequency.
3. Print the discovered start activities and end activities.


In [8]:
# ── Task 3: Process Discovery ──────────────────────────────────────────────
# Goal: discover Petri net (Inductive Miner) and DFG; print stats and top edges
# Method: pm4py.discover_petri_net_inductive, pm4py.discover_dfg

net, im, fm = pm4py.discover_petri_net_inductive(event_log)
print("Petri net places:", len(net.places))
print("Petri net transitions:", len(net.transitions))
print("Petri net arcs:", len(net.arcs))

dfg, start_activities, end_activities = pm4py.discover_dfg(event_log)
print("\nDFG edges:", len(dfg))
top5 = sorted(dfg.items(), key=lambda x: x[1], reverse=True)[:5]
for (a, b), count in top5:
    print(f"  ({a}, {b}): {count}")

print("\nStart activities:", start_activities)
print("End activities:", end_activities)

Petri net places: 10
Petri net transitions: 12
Petri net arcs: 24

DFG edges: 9
  (Out for Delivery, Delivery Attempted): 195
  (Order Placed, Packed): 150
  (Packed, Shipped): 150
  (Shipped, In Transit): 150
  (In Transit, Out for Delivery): 150

Start activities: {'Order Placed': 150}
End activities: {'Delivered': 149, 'Returned to Warehouse': 1}


### Task 4 — Visualization & Export (15 pts)
1. Save a visualization of the Petri net to `petri_net_day187.png`.
2. Save a visualization of the DFG to `dfg_day187.png`.
3. Export a one-row-per-case summary CSV `case_summary_day187.csv` with columns:
   `case_id, carrier, region, num_events, throughput_hours` (throughput = last timestamp − first
   timestamp, in hours, for that case).


In [9]:
# ── Task 4: Visualization & Export ─────────────────────────────────────────
# Goal: save Petri net and DFG images, export case summary CSV
# Method: pm4py.save_vis_petri_net, pm4py.save_vis_dfg, groupby with agg

pm4py.save_vis_petri_net(net, im, fm, "petri_net_day187.png")
pm4py.save_vis_dfg(dfg, start_activities, end_activities, "dfg_day187.png")

case_summary = (
    df.groupby('case_id')
      .agg(
          carrier=('carrier', 'first'),
          region=('region', 'first'),
          num_events=('activity', 'count'),
          throughput_hours=('timestamp', lambda x: (x.max() - x.min()).total_seconds() / 3600)
      )
      .reset_index()
)
case_summary.to_csv("case_summary_day187.csv", index=False)
case_summary.head()

,case_id,carrier,region,num_events,throughput_hours
0,SHIP0001,XpressBees,East,13,85.515490
1,SHIP0002,DTDC,West,7,43.812529
2,SHIP0003,Ecom Express,East,7,41.584325
3,SHIP0004,DTDC,East,7,39.553104
4,SHIP0005,BlueDart,East,7,49.470054


### Task 5 — NRA Business Insight Report (20 pts)
Write exactly 3 bullets in **Number + Reason + Action** format, each grounded in a value you printed
above. Reason = causal mechanism only (stop there — no outcome restatement, no action bleeding in).
Action = specific and committed (name the carrier/threshold/owner — no "could" or "consider").


In [17]:
# ── Task 5: NRA Business Insight Report ────────────────────────────────────
# Goal: 3 bullets with Number (from printed outputs), Reason (causal mechanism only), Action (specific, committed)

# --- Prints that compute the numbers used below ---
print("\n--- Extra computation for NRA ---")
deviant_cases = df[df['activity'] == 'Delivery Failed']['case_id'].unique()
deviant_event_counts = case_summary[case_summary['case_id'].isin(deviant_cases)]['num_events']
clean_event_counts = case_summary[~case_summary['case_id'].isin(deviant_cases)]['num_events']
avg_deviant_events = deviant_event_counts.mean()
avg_clean_events = clean_event_counts.mean()
extra_events = avg_deviant_events - avg_clean_events
print(f"Avg events per deviant case: {avg_deviant_events:.2f}")
print(f"Avg events per clean case: {avg_clean_events:.2f}")
print(f"Extra events per deviant case: {extra_events:.2f}")

carrier_throughput = case_summary.groupby('carrier')['throughput_hours'].mean().round(2)
print("\nAverage throughput hours per carrier:")
print(carrier_throughput.sort_values(ascending=False))
worst_carrier_throughput = carrier_throughput.idxmax()
worst_throughput_val = carrier_throughput.max()
print(f"Worst carrier throughput: {worst_carrier_throughput} – {worst_throughput_val:.2f} hrs")
# -------------------------------------------------

nra_report = f'''
BULLET 1 (Process structure):
Number: Only 75.3% of shipments follow the clean 7‑step happy path (113 of 150 cases)
Reason: The majority of shipments follow the standard sequence without delivery failures because most carriers (BlueDart 90%, Delhivery 85%, Ecom Express 80%, DTDC 78%) have high success rates, allowing the process to complete without deviations.
Action: Mandate that carriers with success rate <80% (XpressBees, DTDC) submit a corrective action plan within 2 weeks; renegotiate SLAs to penalise failure rates above 15%.

BULLET 2 (Rework/deviation):
Number: 37 cases (24.7%) contain at least one 'Delivery Failed' event, averaging {extra_events:.2f} extra events per deviant case compared to clean cases
Reason: Carriers with high failure rates (XpressBees 57.6%, Ecom Express 22.2%) cause repeated delivery attempts.
Action: Shift 60% of XpressBees volume to Delhivery (failure rate 8.3%) effective next month; implement a real‑time alert when a case has >2 delivery attempts to trigger manual intervention.

BULLET 3 (Carrier/throughput):
Number: {worst_carrier_throughput} has the highest average throughput at {worst_throughput_val:.2f} hours and the highest rework rate at 57.58%
Reason: {worst_carrier_throughput} has the lowest underlying success rate (62%) among all carriers, which forces more delivery attempts.
Action: Split {worst_carrier_throughput} volume across two regional hubs (North and South) by Q4; cap its share of total volume at 15% until its failure rate falls below 15%.
'''
print(nra_report)


--- Extra computation for NRA ---
Avg events per deviant case: 10.68
Avg events per clean case: 7.00
Extra events per deviant case: 3.68

Average throughput hours per carrier:
carrier
XpressBees      55.03
DTDC            44.92
Delhivery       44.17
Ecom Express    43.35
BlueDart        42.16
Name: throughput_hours, dtype: float64
Worst carrier throughput: XpressBees – 55.03 hrs

BULLET 1 (Process structure):
Number: Only 75.3% of shipments follow the clean 7‑step happy path (113 of 150 cases)
Reason: The majority of shipments follow the standard sequence without delivery failures because most carriers (BlueDart 90%, Delhivery 85%, Ecom Express 80%, DTDC 78%) have high success rates, allowing the process to complete without deviations.
Action: Mandate that carriers with success rate <80% (XpressBees, DTDC) submit a corrective action plan within 2 weeks; renegotiate SLAs to penalise failure rates above 15%.

BULLET 2 (Rework/deviation):
Number: 37 cases (24.7%) contain at least one 'De

### BONUS (+10★) — Carrier Rework Rate
For each carrier, compute the % of its cases that contain at least one `Delivery Failed` event. Sort
descending and identify the worst carrier by name.


In [11]:
# ── Bonus: Carrier Rework Rate ─────────────────────────────────────────────
# Goal: compute % of cases with Delivery Failed per carrier, identify worst
# Method: groupby carrier, count distinct cases with failed events

carrier_cases = df.groupby('carrier')['case_id'].nunique()
fail_cases = df[df['activity'] == 'Delivery Failed'].groupby('carrier')['case_id'].nunique()
rework_rate = (fail_cases / carrier_cases * 100).sort_values(ascending=False)
print(rework_rate.round(2))
worst = rework_rate.idxmax()
print("Worst carrier:", worst)

carrier
XpressBees      57.58
Ecom Express    22.22
DTDC            17.14
BlueDart        12.90
Delhivery        8.33
Name: case_id, dtype: float64
Worst carrier: XpressBees


## SECTION 4: Answer Key (expected output — do not peek before submitting)

**T1 — Extract & Profile**
```
Raw rows: 1186
Unique cases: 150
Unique activities: 9
Activities: ['Delivered', 'Delivery Attempted', 'Delivery Failed', 'In Transit', 'Order Placed',
             'Out for Delivery', 'Packed', 'Returned to Warehouse', 'Shipped']
Date range: 2026-04-01 09:07:14 -> 2026-05-03 11:29:46
Avg events/case: 7.91
```

**T2 — Variant Analysis**
```
Total unique variants: 4
Top variant: ('Order Placed', 'Packed', 'Shipped', 'In Transit', 'Out for Delivery',
              'Delivery Attempted', 'Delivered')
Top variant share: 113 cases (75.3%)
Deviant (rework) cases: 37 (24.7%)
```

**T3 — Process Discovery**
```
Petri net places: 10
Petri net transitions: 12
Petri net arcs: 24

DFG edges: 9
Top 5 directly-follows relations:
  ('Out for Delivery', 'Delivery Attempted'): 195
  ('Order Placed', 'Packed'): 150
  ('Packed', 'Shipped'): 150
  ('Shipped', 'In Transit'): 150
  ('In Transit', 'Out for Delivery'): 150

Start activities: {'Order Placed': 150}
End activities: {'Delivered': 149, 'Returned to Warehouse': 1}
```
*(Exact place/transition/arc counts can vary ±1-2 depending on pm4py version — the grading tolerance
accounts for this. What matters is the top-5 DFG ranking and the start/end activities, which are
version-independent.)*

**T4 — Visualization & Export**
Two PNGs saved (petri net + DFG, both should render without errors). `case_summary_day187.csv`
should have 150 rows; avg `throughput_hours` ≈ 46.17, median ≈ 43.56, max ≈ 94.49.

**BONUS — Carrier Rework Rate**
```
XpressBees      57.58%
Ecom Express    22.22%
DTDC            17.14%
BlueDart        12.90%
Delhivery        8.33%
```
Worst carrier: **XpressBees** — consistent with the Day 181–186 finding that XpressBees is your
worst-performing carrier (11.42-day avg transit there; 57.58% rework rate here). Two independent
analyses on two different cuts of LogiTrack India data point at the same carrier — that consistency is
itself worth stating to a client.


## SECTION 5: Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| T1 — Extract & Profile | 15 | 3 pts each: rows, cases, activities+list, date range, avg events/case |
| T2 — Variant Analysis | 20 | 5 pts each: variant count, top variant identified, top variant %, deviant count/% |
| T3 — Process Discovery | 20 | 5 pts each: Petri net stats, DFG edge count, top-5 DFG ranking, start/end activities |
| T4 — Visualization & Export | 15 | 5 pts each: Petri net PNG saved & valid, DFG PNG saved & valid, case_summary CSV correct (150 rows, correct throughput calc) |
| T5 — NRA Report | 20 | ~6-7 pts per bullet: Number matches printed output, Reason is causal mechanism only (no outcome restatement), Action is specific/committed (no hedging) |
| **Total** | **90** | |
| Bonus — Carrier Rework Rate | +10★ | Correct % per carrier, correct worst-carrier identification (XpressBees) |

**Technical vs Communication split:** T1-T4 and the bonus are scored as technical (right output = full
credit). T5 is scored as communication — a technically correct number wrapped in a vague or drifting
Reason/Action still loses points, because in client work the insight is worthless if it can't be stated
cleanly.


---
## Interview Framing

**"Walk me through a process mining project you've done."**

> "I took a raw shipment event log — case ID, activity, timestamp — and reconstructed the actual delivery
> process using pm4py's Inductive Miner. Variant analysis showed 75% of shipments followed the clean
> 7-step happy path, but 25% looped through failed delivery attempts. Cross-referencing that against
> carrier gave a clear signal: one carrier accounted for the majority of the rework, both in failure rate
> and average throughput time. That's the kind of finding you can't get from a static flowchart — it only
> shows up when you mine the actual event timestamps."

## Key Takeaway
**A directly-follows graph tells you where volume flows; a Petri net tells you what's structurally
possible. Variant analysis is the fastest triage step — before building any formal model, know what % of
your cases are "happy path" vs "exception," because that ratio alone tells a client how healthy their
process is.**

---
**Next session: Day 188 — Process Mining with pm4py Part 2 (conformance checking, fitness/precision
metrics, KPI dashboard).**
